# TP Sistemas Inteligentes — Grupo 08 (UTN FRBA, 2C 2026)

Clasificación de variedades de arroz a partir de imágenes (*Rice Image Dataset*) con **redes neuronales convolucionales** (Keras).

**Cómo ejecutarlo en Google Colab**

1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
2. Menú *Entorno de ejecución → Ejecutar todas*.
3. Cuando lo pida la primera celda, elegir el archivo `TP2_Grupo08_repositorio.zip`.
4. Al terminar se descarga solo el archivo `resultados_cnn.zip` con los gráficos y las métricas.

## 1. Librerías y datos

In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

keras.utils.set_random_seed(42)

# Si las imagenes no estan, se pide el zip del repositorio y se descomprime
if not os.path.exists("data"):
    from google.colab import files
    print("Elegir el archivo TP2_Grupo08_repositorio.zip")
    subido = files.upload()
    shutil.unpack_archive(list(subido.keys())[0], ".")
    os.chdir("tp2-sistint-grupo08")
print("Carpeta de trabajo:", os.getcwd())

## 2. Carga de las imágenes

In [ ]:
TAMANIO = (64, 64)   # las imagenes se reducen de 250x250 a 64x64 pixeles (en color)

# 90 % de data/train para entrenar y 10 % para validacion
train_ds = keras.utils.image_dataset_from_directory(
    "data/train", image_size=TAMANIO, batch_size=32,
    validation_split=0.1, subset="training", seed=42)
val_ds = keras.utils.image_dataset_from_directory(
    "data/train", image_size=TAMANIO, batch_size=32,
    validation_split=0.1, subset="validation", seed=42)
test_ds = keras.utils.image_dataset_from_directory(
    "data/test", image_size=TAMANIO, batch_size=32, shuffle=False)

CLASES = train_ds.class_names
print(CLASES)

## 3. Ejemplo de una imagen de cada variedad

In [ ]:
os.makedirs("resultados", exist_ok=True)
fig, ejes = plt.subplots(2, 5, figsize=(7, 3.2))
for i, clase in enumerate(CLASES):
    archivo = f"data/train/{clase}/" + sorted(os.listdir(f"data/train/{clase}"))[0]
    ejes[0, i].imshow(keras.utils.load_img(archivo))
    ejes[1, i].imshow(keras.utils.load_img(archivo, target_size=TAMANIO, interpolation="bilinear"))
    ejes[0, i].set_title(clase)
    ejes[0, i].axis("off")
    ejes[1, i].axis("off")
ejes[0, 0].text(-40, 125, "Original\n250x250", ha="right", va="center")
ejes[1, 0].text(-10, 32, "Entrada\n64x64", ha="right", va="center")
plt.tight_layout()
plt.savefig("resultados/ejemplos.png", dpi=300)
plt.show()

## 4. Definición de los tres modelos (cambia la cantidad de capas convolucionales)

In [ ]:
def modelo_1():
    return keras.Sequential([
        keras.Input(shape=(64, 64, 3)),
        layers.Rescaling(1 / 255),                      # pixeles entre 0 y 1
        layers.Conv2D(8, 3, activation="relu"),         # 8 filtros de 3x3
        layers.MaxPooling2D(),                          # reduce la imagen a la mitad
        layers.Flatten(),
        layers.Dense(5, activation="softmax"),          # una salida por variedad
    ])


def modelo_2():
    return keras.Sequential([
        keras.Input(shape=(64, 64, 3)),
        layers.Rescaling(1 / 255),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(5, activation="softmax"),
    ])


def modelo_3():
    return keras.Sequential([
        keras.Input(shape=(64, 64, 3)),
        layers.Rescaling(1 / 255),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(5, activation="softmax"),
    ])


modelos = {"Modelo 1": modelo_1, "Modelo 2": modelo_2, "Modelo 3": modelo_3}

## 5. Entrenamiento y evaluación

In [ ]:
def predecir(red, datos):
    # Devuelve las clases reales y las predichas por la red para un conjunto de imagenes
    reales, predichas = [], []
    for imagenes, etiquetas in datos:
        reales.extend(np.array(etiquetas))
        predichas.extend(np.argmax(red.predict_on_batch(imagenes), axis=1))
    return np.array(reales), np.array(predichas)


resultados = {}
for nombre, crear in modelos.items():
    print("=" * 60)
    print(nombre)
    red = crear()
    red.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    red.summary()

    # Se detiene si el error de validacion no mejora durante 5 epocas
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    historia = red.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[parada], verbose=2)

    resultados[nombre] = {"red": red, "historia": historia.history,
                          "train": predecir(red, train_ds),
                          "val": predecir(red, val_ds),
                          "test": predecir(red, test_ds)}
    real, pred = resultados[nombre]["test"]
    print(classification_report(real, pred, target_names=CLASES, digits=4))

## 6. Gráfico del error (Loss) durante el entrenamiento

In [ ]:
plt.rcParams["font.size"] = 10
for nombre, r in resultados.items():
    h = r["historia"]
    epocas = range(1, len(h["loss"]) + 1)
    plt.figure(figsize=(6, 2.8))
    plt.plot(epocas, h["loss"], label="Entrenamiento")
    plt.plot(epocas, h["val_loss"], label="Validación")
    plt.title(f"Error (Loss) durante el entrenamiento - {nombre}")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(f"resultados/loss_{nombre.replace(' ', '_')}.png", dpi=300)
    plt.show()

## 7. Matrices de confusión (entrenamiento y prueba)

In [ ]:
for nombre, r in resultados.items():
    fig, ejes = plt.subplots(1, 2, figsize=(7, 3.6))
    for eje, conjunto, titulo in [(ejes[0], "train", "Entrenamiento"), (ejes[1], "test", "Prueba")]:
        real, pred = r[conjunto]
        ConfusionMatrixDisplay.from_predictions(real, pred, display_labels=CLASES, cmap="Blues",
                                                colorbar=False, ax=eje, text_kw={"fontsize": 9})
        eje.set_title(titulo)
        eje.set_xlabel("Clase predicha")
        eje.set_ylabel("Clase real")
        eje.tick_params(axis="x", labelrotation=90)
    plt.tight_layout()
    plt.savefig(f"resultados/matriz_{nombre.replace(' ', '_')}.png", dpi=300)
    plt.show()

## 8. Guardado de los resultados (se descarga `resultados_cnn.zip`)

In [ ]:
salida = {"clases": CLASES, "keras": keras.__version__, "modelos": {}}
for nombre, r in resultados.items():
    info = {"parametros": int(r["red"].count_params()),
            "epocas": len(r["historia"]["loss"]),
            "historia": {k: [float(v) for v in valores] for k, valores in r["historia"].items()}}
    for conjunto in ["train", "val", "test"]:
        real, pred = r[conjunto]
        info[conjunto] = {"cantidad": len(real),
                          "matriz": confusion_matrix(real, pred).tolist(),
                          "reporte": classification_report(real, pred, target_names=CLASES, output_dict=True)}
    salida["modelos"][nombre] = info

with open("resultados/resultados_cnn.json", "w") as archivo:
    json.dump(salida, archivo, indent=1, default=float)

shutil.make_archive("resultados_cnn", "zip", "resultados")
try:
    from google.colab import files
    files.download("resultados_cnn.zip")
except ImportError:
    print("Resultados guardados en resultados_cnn.zip")